In [68]:
import pandas as pd
import torch
import random
from tqdm.auto import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import ast


In [69]:
test_df = pd.read_csv("datasets_llm_multitask_df.csv")
ps = test_df[test_df["task"] == "personalized_summarization"]

valid_queries = (
    ps["query_text_id"]
    .value_counts()
    .loc[lambda x: x >= 3]
    .index
)

ps_valid = ps[ps["query_text_id"].isin(valid_queries)]

selected = []
count = 0
for qid, group in ps_valid.groupby("query_text_id"):
    if count >= 150:
        break
    if count + len(group) <= 100:
        selected.append(group)
        count += len(group)

ps_150 = pd.concat(selected)

df_rest = test_df.drop(ps_150.index)

SEED = 42

task_dfs = {}
task_dfs["personalized_summarization"] = ps_150

for task in ["next_item_prediction", "next_user_prediction", "dialogue_generation"]:
    subset = df_rest[df_rest["task"] == task]
    task_dfs[task] = subset.sample(n=150, random_state=SEED)

test_df = (
    pd.concat(task_dfs.values())
      .sample(frac=1, random_state=SEED)
      .reset_index(drop=True)
)

test_df.to_csv("filtered_test.csv")

In [ ]:
test_df

In [ ]:
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

def llm_generate(prompt, max_new_tokens=128):
    out = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    text = out[0]["generated_text"]
    return text[len(prompt):].strip() if text.startswith(prompt) else text.strip()


In [72]:
def safe_parse(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

test_df["text_nodes"] = test_df["text_nodes"].apply(safe_parse)
test_df["interactions"] = test_df["interactions"].apply(safe_parse)

docs = pd.read_csv("datasets_documents.csv")
resp = pd.read_csv("response.csv")
dial = pd.read_csv("dialogue.csv")
summ = pd.read_csv("summaries (1).csv")

doc_map  = dict(zip(docs["NewsID"], docs["NewsBody"]))
resp_map = dict(zip(resp["response_id"], resp["response_text"]))
dial_map = dict(zip(dial["dialogue_id"], dial["dialogue_text"]))
summ_map = dict(zip(summ["SummID"], summ["Summary"]))


In [73]:
def resolve_node(node_id):
    if node_id.startswith("d-"):  return doc_map.get(node_id, "")
    if node_id.startswith("r-"):  return resp_map.get(node_id, "")
    if node_id.startswith("dl-"): return dial_map.get(node_id, "")
    if node_id.startswith("s-"):  return summ_map.get(node_id, "")
    return ""

def build_history(row):
    history = []
    for node, act in zip(row["text_nodes"], row["interactions"]):
        text = resolve_node(node)
        history.append(text)
    return "\n".join(history)


In [ ]:
up_df = test_df[test_df["task"]=="next_user_prediction"].copy()

user_histories = {}

for uid, g in tqdm(up_df.groupby("userid")):
    texts = []
    for _, row in g.iterrows():
        texts.append(build_history(row))
    user_histories[uid] = "\n".join(texts[:300])


In [ ]:
persona_rows = []

for uid, hist in tqdm(user_histories.items()):
    prompt = f"""
Write a generic persona from this history.

Rules:
- 15 to 30 words
- No names, places, numbers, or quotes
- Only broad preferences and style

History:
{hist}

Persona:
""".strip()

    raw = llm_generate(prompt, max_new_tokens=40)
    persona = " ".join(raw.split()[:30])

    persona_rows.append({"userid": uid, "persona": persona})

persona_df = pd.DataFrame(persona_rows)
persona_df.to_csv("user_personas.csv", index=False)

persona_map = dict(zip(persona_df["userid"], persona_df["persona"]))
user_ids = list(persona_map.keys())


In [ ]:
user_pred_results = []

for _, row in tqdm(up_df.iterrows(), total=len(up_df)):
    true_uid = row["userid"]
    wrong_uid = random.choice([u for u in user_ids if u != true_uid])

    if random.random() < 0.5:
        A_uid, B_uid = true_uid, wrong_uid
    else:
        A_uid, B_uid = wrong_uid, true_uid

    prompt = f"""
Conversation:
{build_history(row)}

User A:
{persona_map[A_uid]}

User B:
{persona_map[B_uid]}

Which user wrote the conversation?

Output only A or B.
Answer:
"""

    raw = llm_generate(prompt, max_new_tokens=5).strip()

    if raw == "A":
        pred = A_uid
    else:
        pred = B_uid
    
    user_pred_results.append({
        "userid": true_uid,
        "prediction": pred,
        "correct": (pred == true_uid)
    })


In [ ]:
pred_df = pd.DataFrame(user_pred_results)
pred_df.to_csv("deepseek_user_predictions.csv", index=False)

accuracy = pred_df["correct"].mean() * 100
print(f"Speaker Identification Accuracy DEEPSEEK-32B: {accuracy:.2f}%")


In [83]:
import pandas as pd
import random
from tqdm.auto import tqdm

persona_df = pd.read_csv("user_personas.csv")
persona_map = dict(zip(persona_df["userid"], persona_df["persona"]))
user_ids = list(persona_map.keys())


In [84]:
docs = pd.read_csv("datasets_documents.csv")
resp = pd.read_csv("response.csv")
dial = pd.read_csv("dialogue.csv")
summ = pd.read_csv("summaries (1).csv")

doc_map  = dict(zip(docs["NewsID"], docs["NewsBody"]))
resp_map = dict(zip(resp["response_id"], resp["response_text"]))
dial_map = dict(zip(dial["dialogue_id"], dial["dialogue_text"]))
summ_map = dict(zip(summ["SummID"], summ["Summary"]))


In [85]:
def resolve_node(node_id):
    if node_id.startswith("d-"):  return doc_map.get(node_id, "")
    if node_id.startswith("r-"):  return resp_map.get(node_id, "")
    if node_id.startswith("dl-"): return dial_map.get(node_id, "")
    if node_id.startswith("s-"):  return summ_map.get(node_id, "")
    return ""

def build_history(row):
    history = []
    for node in row["text_nodes"]:
        history.append(resolve_node(node))
    return "\n".join(history)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto"
)

def llm_generate(prompt, max_new_tokens=64):
    out = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    text = out[0]["generated_text"]
    return text[len(prompt):].strip() if text.startswith(prompt) else text.strip()


In [99]:
import numpy as np

N = len(up_df[:5])
rng = np.random.default_rng(42)

wrong_uids = rng.choice(user_ids, size=N)
swap = rng.random(N) < 0.5

rows = up_df[:5].reset_index(drop=True)


In [100]:
prompts = []
meta = []

for i, row in rows.iterrows():
    true_uid = row["userid"]
    wrong_uid = wrong_uids[i] if wrong_uids[i] != true_uid else random.choice([u for u in user_ids if u != true_uid])

    if swap[i]:
        A_uid, B_uid = wrong_uid, true_uid
    else:
        A_uid, B_uid = true_uid, wrong_uid

    prompt = f"""
Conversation:
{build_history(row)}

User A:
{persona_map[A_uid]}

User B:
{persona_map[B_uid]}

Which user wrote the conversation?
Output only A or B.

Answer:
"""

    prompts.append(prompt)
    meta.append((true_uid, A_uid, B_uid))


In [101]:
from tqdm.auto import tqdm

def batched_generate(prompts, batch_size=8, max_new_tokens=1):
    outputs = []

    for i in tqdm(range(0, len(prompts), batch_size), desc="Running Qwen-32B"):
        batch = prompts[i:i+batch_size]

        out = generator(
            batch,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

        outputs.extend(out)

    return outputs


In [ ]:
outputs = batched_generate(
    prompts,
    batch_size=1,
    max_new_tokens=1
)


In [ ]:
outputs

In [ ]:
user_pred_results = []

for out, (true_uid, A_uid, B_uid) in zip(outputs, meta):
    text = out[0]["generated_text"].strip()   # <-- fix here

    if text.endswith("A"):
        pred = A_uid
    elif text.endswith("B"):
        pred = B_uid
    else:
        pred = "INVALID"

    user_pred_results.append({
        "userid": true_uid,
        "prediction": pred,
        "correct": (pred == true_uid)
    })


In [ ]:
pred_df = pd.DataFrame(user_pred_results)
pred_df.to_csv("qwen32b_user_predictions.csv", index=False)

accuracy = pred_df["correct"].mean() * 100
print(f"Speaker Identification Accuracy QWEN-32B: {accuracy:.2f}%")


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

MODEL_ID = "microsoft/phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto"
)

def llm_generate(prompt, max_new_tokens=64):
    out = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    text = out[0]["generated_text"]
    return text[len(prompt):].strip() if text.startswith(prompt) else text.strip()


In [ ]:
from tqdm.auto import tqdm

def batched_generate(prompts, batch_size=16):
    outputs = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="Running Phi reader"):
        batch = prompts[i:i+batch_size]
        out = generator(batch, max_new_tokens=1, do_sample=False)
        outputs.extend(out)
    return outputs

outputs = batched_generate(prompts, batch_size=16)


In [112]:
user_pred_results = []

for out, (true_uid, A_uid, B_uid) in zip(outputs, meta):
    text = out[0]["generated_text"].strip()

    if text.endswith("A"):
        pred = A_uid
    elif text.endswith("B"):
        pred = B_uid
    else:
        pred = "INVALID"

    user_pred_results.append({
        "userid": true_uid,
        "prediction": pred,
        "correct": (pred == true_uid)
    })


In [ ]:
import pandas as pd

pred_df = pd.DataFrame(user_pred_results)
pred_df.to_csv("phi_reader_user_predictions.csv", index=False)

accuracy = pred_df["correct"].mean() * 100
print(f"Speaker Identification Accuracy (Phi reader): {accuracy:.2f}%")
